In [1]:
import pickle
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Ubicación segura de data/ y submission/
cwd = Path.cwd()
if (cwd / "data").is_dir() and (cwd / "submission").is_dir():
    ACTIVITY_DIR = cwd
elif (cwd.parent / "data").is_dir() and (cwd.parent / "submission").is_dir():
    ACTIVITY_DIR = cwd.parent
else:
    ACTIVITY_DIR = cwd / "P200_regresion_basica"

DATA_FILE = ACTIVITY_DIR / "data" / "auto_mpg.csv"
SUBMISSION_DIR = ACTIVITY_DIR / "submission"

# 2. Carga y limpieza de datos
dataset = pd.read_csv(DATA_FILE)
dataset = dataset.dropna()
dataset["Origin"] = dataset["Origin"].astype("category")

train_dataset, test_dataset = train_test_split(
    dataset,
    train_size=0.8,
    random_state=0,
)

train_features = train_dataset.copy()
train_labels = train_features.pop("MPG")

test_features = test_dataset.copy()
test_labels = test_features.pop("MPG")

test_results = {}

# 3. Modelo 1: Regresión lineal simple (Horsepower)
horsepower_scaler = StandardScaler()
train_horsepower = train_features[["Horsepower"]]
test_horsepower = test_features[["Horsepower"]]

standarized_train_horsepower = horsepower_scaler.fit_transform(train_horsepower)
standarized_test_horsepower = horsepower_scaler.transform(test_horsepower)

horsepower_model = LinearRegression()
horsepower_model.fit(standarized_train_horsepower, train_labels)

test_results["horsepower_model"] = mean_squared_error(
    y_true=test_labels,
    y_pred=horsepower_model.predict(standarized_test_horsepower),
)

# 4. Modelo 2: Regresión lineal múltiple con preprocesador
numeric_features = train_features.select_dtypes(include="number").columns.tolist()
categorical_features = ["Origin"]

features_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "origin",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ],
)

standarized_train_features = features_preprocessor.fit_transform(train_features)
standarized_test_features = features_preprocessor.transform(test_features)

linear_model = LinearRegression()
linear_model.fit(standarized_train_features, train_labels)

test_results["linear_model"] = mean_squared_error(
    y_true=test_labels,
    y_pred=linear_model.predict(standarized_test_features),
)

# 5. Modelo 3: Regresión lineal flexible (término cuadrático + interacción)
train_features_flexible = train_features.copy()
test_features_flexible = test_features.copy()

for features in [train_features_flexible, test_features_flexible]:
    features["Horsepower_squared"] = features["Horsepower"] ** 2
    features["Weight_x_Horsepower"] = features["Weight"] * features["Horsepower"]

flexible_numeric_features = train_features_flexible.select_dtypes(
    include="number"
).columns.tolist()

flexible_features_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), flexible_numeric_features),
        (
            "origin",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ],
)

flexible_train_features = flexible_features_preprocessor.fit_transform(
    train_features_flexible
)
flexible_test_features = flexible_features_preprocessor.transform(
    test_features_flexible
)

linear_flexible_model = LinearRegression()
linear_flexible_model.fit(flexible_train_features, train_labels)

test_results["linear_flexible_model"] = mean_squared_error(
    y_true=test_labels,
    y_pred=linear_flexible_model.predict(flexible_test_features),
)

# 6. Modelo 4: Red neuronal (MLP) con una sola variable
mlp_horsepower = MLPRegressor(
    max_iter=10000,
    hidden_layer_sizes=(64, 64),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    validation_fraction=0.2,
    early_stopping=True,
    random_state=0,
)
mlp_horsepower.fit(standarized_train_horsepower, train_labels)

test_results["mlp_horsepower"] = mean_squared_error(
    y_true=test_labels,
    y_pred=mlp_horsepower.predict(standarized_test_horsepower),
)

# 7. Modelo 5: Red neuronal (MLP) con todas las variables
mlp = MLPRegressor(
    max_iter=10000,
    hidden_layer_sizes=(64, 64),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    validation_fraction=0.2,
    early_stopping=True,
    random_state=0,
)
mlp.fit(standarized_train_features, train_labels)

test_results["mlp"] = mean_squared_error(
    y_true=test_labels,
    y_pred=mlp.predict(standarized_test_features),
)

# 8. Guardar modelos, preprocesadores y comparación en submission/
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

with open(SUBMISSION_DIR / "mlp.pkl", "wb") as file:
    pickle.dump(mlp, file)

with open(SUBMISSION_DIR / "features_preprocessor.pkl", "wb") as file:
    pickle.dump(features_preprocessor, file)

with open(SUBMISSION_DIR / "linear_flexible_model.pkl", "wb") as file:
    pickle.dump(linear_flexible_model, file)

with open(SUBMISSION_DIR / "flexible_features_preprocessor.pkl", "wb") as file:
    pickle.dump(flexible_features_preprocessor, file)

comparison_df = pd.DataFrame(
    {
        "model": list(test_results),
        "test_mse": list(test_results.values()),
    }
)
comparison_df.to_csv(SUBMISSION_DIR / "model_comparison.csv", index=False)

print(comparison_df)
print("\nArchivos guardados en:", SUBMISSION_DIR)

                   model   test_mse
0       horsepower_model  22.026387
1           linear_model  10.022425
2  linear_flexible_model   7.441404
3         mlp_horsepower  15.599045
4                    mlp   7.355843

Archivos guardados en: c:\Users\samu6\Documents\GitHub\std-samuel-lopera-jaramillo\P200_regresion_basica\submission
